In [1]:
import numpy as np
import pandas as pd
import os
import random
import re


import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns



import ete3 as ete

import scipy.stats as stats
from functools import *

import Bio
from Bio import Entrez
from Bio import SeqIO
from Bio.SeqUtils import GC
from Bio.SeqFeature import SeqFeature, FeatureLocation
from Bio import Seq



sns.set_context("paper")
%matplotlib inline



# Key files for this script

This script parses all Genbank files from the Refseq viral database. It only parses files that show up in the Virus Host DB as having human hosts. This script identifies all annotated CDS and mature peptides and identifies all ORFs that show up in the genome. The minimum ORF length is 8aa for this script. Key files produced from this script include:

- refseq_all_orf_df_assigned.csv  
File contains all annotated CDS and mature peptides. Also contains Unannotated ORFs. The file removes "redundancy", so all ORFs identified with theoretical translations that match annotated CDSs are removed to only include the annotated ones

- genome_df_raw.csv
File contains info about the accession

In [7]:
#Specify translation table
#more info: https://www.ncbi.nlm.nih.gov/Taxonomy/Utils/wprintgc.cgi#SG1



def find_orfs_with_trans(seq, start_codons=['ATG', 'CTG'], stop_codons = ['TAA','TAG', 'TGA'],
                        min_len  = (8*3)):
    
    '''
    Function that finds all ORFs in the 6 possible reading frames of a sequence. 
    The function uses the standard codon table.
    Can modify start codons to include more alternative start codons. 
    'CTG' start codon is still encoded as Met
    Returns a list of tuples for all ORFs identified.
    Tuple is in the following order:
        (start, end, strand, seq_nt, seq_aa)
        
    For now, skip ambigous sequences
    min_len corresponds to bp including start and stop codon

    '''
    
#     trans_table = Bio.Data.CodonTable.CodonTable(nucleotide_alphabet=Bio.Data.CodonTable.standard_dna_table.nucleotide_alphabet, 
#                                                  protein_alphabet=Bio.Data.CodonTable.standard_dna_table.protein_alphabet,
#                                                  forward_table=Bio.Data.CodonTable.standard_dna_table.forward_table,
#                                                  back_table=Bio.Data.CodonTable.standard_dna_table.back_table,
#                                                  start_codons=start_codons,
#                                                  stop_codons=Bio.Data.CodonTable.standard_dna_table.stop_codons
#                                                 )
    #trans_table =Bio.Data.CodonTable.standard_dna_table
    
    seq = Seq.Seq(seq.upper())
    seq_len = len(seq)

    #parse start and stop codons:
    start_codons = [f"{codon}" for codon in start_codons]
    start_code= '|'.join(start_codons)
    stop_codons = [f"{codon}" for codon in stop_codons]
    stop_code = '|'.join(stop_codons)

    #Define string pattern for ORF 
    pattern = re.compile(f'(?=(({start_code})(?:...)*?)(?=({stop_code})))')

    
    df = {
        'location':[],
        'start': [],
         'end': [],
          'strand': [],
         'seq_nt': [],
         'seq_aa': [], 
         'length_aa': [],
        'start_codon': []
         }
    
    #loop through strands    
    for strand, nuc in [(+1, seq), (-1, seq.reverse_complement())]:

        #find orfs
        orfs = [o[0]+o[2] for o in pattern.findall(str(nuc)) if ((len(o[0]) >= min_len) & (len(o[0]) % 3 == 0))]
        starts = []
        ends = []
        seqs_nt = [] 
        seqs_aa = []
        lengths_aa = []
        locations = []
        start_codons =[]

        for i in range(len(orfs)):
            try:
                #check if sequence contains only ATCG
                o = orfs[i]
                start = nuc.find(o)
                end = start + len(o)
                seq_nt = str(nuc[start:end])
                
                       
                       
                if pd.Series(pd.Series(list(seq_nt)).unique()).isin(['A','T','C','G']).all():

                    if seq_nt[0:3] == 'CTG':
                        seq_nt_trans = 'ATG' + seq_nt[3:]
                    else:
                        seq_nt_trans = seq_nt

                    s = str(Seq.Seq(seq_nt_trans).translate())[0:-1]
                    
                    starts.append(start)
                    ends.append(end)
                    seqs_nt.append(seq_nt)
                    seqs_aa.append(s)
                    lengths_aa.append(len(s))
                    start_codons.append(seq_nt[0:3])

            except Bio.Data.CodonTable.TranslationError:
                print('Ambiguous Codon Detected for seq:')
                print(seq_nt)
                s = 'Ambiguous Codon Detected'
                seqs_aa.append(s)
                lengths_aa.append(0)

        #modify to reverse complement sequence postion 
        if strand == -1:
            ends_new = [len(seq)-s for s in starts]
            starts_new = [len(seq)-e for e in ends]
        else:
            ends_new = ends
            starts_new = starts

        locations = [str(FeatureLocation(starts_new[i], ends_new[i], strand = strand))  for i in range(len(starts))] 

        df['start'] =df['start'] + starts_new
        df['start_codon'] =df['start_codon'] + start_codons
        df['end'] = df['end'] + ends_new
        df['seq_nt'] = df['seq_nt'] + seqs_nt
        df['seq_aa'] = df['seq_aa'] + seqs_aa 
        df['strand'] = df['strand'] + [strand]*len(starts)
        df['length_aa'] = df['length_aa'] + lengths_aa
        df['location'] = df['location'] + locations
            
        
    return pd.DataFrame(df).drop_duplicates()

In [8]:
seq = 'CTGNAATGACGACAACAGCGACGGGGACGCCACGATCACTATTAACGCGAGTCTCGGCCTAGCCGCGGGCGACGCCGCTGGCGGCGGCGCTGATCACCACCTGCGGGGCAGCCCGGGCGATTCGCCGCCGCCGATACCTTTCGAGGACGAAAACACGCCCGAGCTGCTGGGCCGGCTCAACGTGTACGAGGTAGCGCGCTTTTCACTGCCGGCTTTTGTCAATCCGCGTCACCAGTATTACTTTCAGATGCTCATTCAGCAGTACGTGCTCAGCCAATACTATATAAAGAAGCATCCGGACCCGGAGCGGATCGATTTCCGCGACCTGCCTACCGTCTACCTGGTCTCGGCCATCTTCCGCGAGCGCGAGGAAAGCGAACTGGGCTGCGAGTTGCTGGCCGGCGGTCGCGTTTTCCACTGCGACCACATCCCGCTCCTGCTCATCGTCACGCCCGTGGTCTTTGACCCTCAGTTTACGCGCCATGCCGTCTCTACCGTGTTAGACCGTTGGAGTCGCGACCTGTCCCGCAAGACGAACCTACCGATATGGGTGCCGAACTCTGCAAACGAATATGTTGTGAGTTCGGTACCACGTCCGTGA'
find_orfs_with_trans(Seq.Seq(seq))


,location,start,end,strand,seq_nt,seq_aa,length_aa,start_codon
0,[5:62](+),5,62,1,ATGACGACAACAGCGACGGGGACGCCACGATCACTATTAACGCGAG...,MTTTATGTPRSLLTRVSA,18,ATG
1,[77:194](+),77,194,1,CTGGCGGCGGCGCTGATCACCACCTGCGGGGCAGCCCGGGCGATTC...,MAAALITTCGAARAIRRRRYLSRTKTRPSCWAGSTCTR,38,CTG
2,[89:194](+),89,194,1,CTGATCACCACCTGCGGGGCAGCCCGGGCGATTCGCCGCCGCCGAT...,MITTCGAARAIRRRRYLSRTKTRPSCWAGSTCTR,34,CTG
3,[100:601](+),100,601,1,CTGCGGGGCAGCCCGGGCGATTCGCCGCCGCCGATACCTTTCGAGG...,MRGSPGDSPPPIPFEDENTPELLGRLNVYEVARFSLPAFVNPRHQY...,166,CTG
4,[163:601](+),163,601,1,CTGCTGGGCCGGCTCAACGTGTACGAGGTAGCGCGCTTTTCACTGC...,MLGRLNVYEVARFSLPAFVNPRHQYYFQMLIQQYVLSQYYIKKHPD...,145,CTG
5,[166:601](+),166,601,1,CTGGGCCGGCTCAACGTGTACGAGGTAGCGCGCTTTTCACTGCCGG...,MGRLNVYEVARFSLPAFVNPRHQYYFQMLIQQYVLSQYYIKKHPDP...,144,CTG
6,[205:601](+),205,601,1,CTGCCGGCTTTTGTCAATCCGCGTCACCAGTATTACTTTCAGATGC...,MPAFVNPRHQYYFQMLIQQYVLSQYYIKKHPDPERIDFRDLPTVYL...,131,CTG
7,[247:601](+),247,601,1,ATGCTCATTCAGCAGTACGTGCTCAGCCAATACTATATAAAGAAGC...,MLIQQYVLSQYYIKKHPDPERIDFRDLPTVYLVSAIFREREESELG...,117,ATG
8,[325:601](+),325,601,1,CTGCCTACCGTCTACCTGGTCTCGGCCATCTTCCGCGAGCGCGAGG...,MPTVYLVSAIFREREESELGCELLAGGRVFHCDHIPLLLIVTPVVF...,91,CTG
9,[340:601](+),340,601,1,CTGGTCTCGGCCATCTTCCGCGAGCGCGAGGAAAGCGAACTGGGCT...,MVSAIFREREESELGCELLAGGRVFHCDHIPLLLIVTPVVFDPQFT...,86,CTG


In [4]:
#gather files

gb_file_dir = "/lab/solexa_weissman/yhc/viral_smORF/data_downloads/RefSeq_viral_sequences/viral_all"
tax_info = pd.read_csv("/lab/solexa_weissman/yhc/viral_smORF/data_downloads/virushostdb/refseq_human_virus_taxinfo.csv")
display(tax_info.head())
human_acc = pd.read_csv("/lab/solexa_weissman/yhc/viral_smORF/data_downloads/virushostdb/refseq_human_virus_accessions.csv")
display(human_acc.head())

out_path = "/lab/solexa_weissman/yhc/viral_smORF/data_analysis/virushostdb/orf_files/"


,virus_name,virus_tax_id,refseq_id_seg,segmented,index,genome_type,genome_composition,superkingdom,phylum,subphylum,...,KEGG_GENOME,KEGG_DISEASE,DISEASE,host_tax_id,host_name,host_lineage,pmid,evidence,sample_type,source_organism
0,Adult diarrheal rotavirus strain J19,335103,"NC_007548,NC_007549,NC_007550,NC_007551,NC_007...",True,1.0,dsRNA,Viruses,Duplornaviricota,NaN,Resentoviricetes,...,NaN,NaN,NaN,9606.0,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,NaN,UniProt,NaN,NaN
1,Alenquer virus,629726,"NC_055330,NC_055331,NC_055332",True,1.0,ssRNA(-),Viruses,Negarnaviricota,Polyploviricotina,Ellioviricetes,...,NaN,NaN,NaN,9606.0,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,21289119,"Literature, RefSeq",NaN,NaN
2,Argentinian mammarenavirus,2169991,"NC_005080,NC_005081",True,1.0,ssRNA(+/-),Viruses,Negarnaviricota,Polyploviricotina,Ellioviricetes,...,T40005,H01541,Argentine hemorrhagic fever,9606.0,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,NaN,UniProt,NaN,NaN
3,Banna virus strain JKT-6423,649604,"NC_004198,NC_004200,NC_004201,NC_004202,NC_004...",True,1.0,dsRNA,Viruses,Duplornaviricota,NaN,Resentoviricetes,...,NaN,NaN,NaN,9606.0,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,NaN,UniProt,NaN,NaN
4,Bayou orthohantavirus,1980459,"NC_038298,NC_038299,NC_038300",True,1.0,ssRNA(-),Viruses,Negarnaviricota,Polyploviricotina,Ellioviricetes,...,T40358,H00390,Hantavirus pulmonary syndrome,9606.0,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,9347961,"Literature, NCBI Virus",NaN,NaN


,refseq_id,virus_name,virus_tax_id
0,NC_007548,Adult diarrheal rotavirus strain J19,335103
1,NC_007549,Adult diarrheal rotavirus strain J19,335103
2,NC_007550,Adult diarrheal rotavirus strain J19,335103
3,NC_007551,Adult diarrheal rotavirus strain J19,335103
4,NC_007552,Adult diarrheal rotavirus strain J19,335103


In [5]:
genome_df = []
cds_df_all = []


for root, dirs, files in os.walk(gb_file_dir):
    for file in files:
        if '.gbff' in file:
            j=0
            gb_file = file
            
            with open(os.path.join(root,gb_file)) as input_handle:
                for i, record in enumerate(SeqIO.parse(input_handle, "genbank")):

                    #if viral genome record has human host, then continue parsing
                    #check if record as human host
                    if record.name in list(human_acc.refseq_id):
                        print(record.name)
                        j+=1
                        
                        g_df = human_acc[human_acc.refseq_id == record.name].reset_index(drop=True)
                        g_df = tax_info[tax_info.virus_tax_id == g_df.virus_tax_id[0]].reset_index(drop=True).merge(g_df, how = 'inner', on =['virus_name', 'virus_tax_id'] )
                        g_df = g_df[['virus_name', 'virus_tax_id', 'refseq_id_seg', 'segmented',
                   'genome_type', 'genome_composition', 'superkingdom',
                   'phylum', 'subphylum', 'class', 'order', 'suborder', 'family',
                   'subfamily', 'genus', 'subgenus', 'species', 'kingdom', 'strain',
                   'serotype', 'clade', 'isolate', 'genotype', 'serogroup', 'virus',
                   'virus_lineage', 'refseq_id_x', 'KEGG_GENOME', 'KEGG_DISEASE',
                   'DISEASE', 'host_tax_id', 'host_name', 'host_lineage', 'pmid',
                   'evidence', 'sample_type', 'source_organism','refseq_id_y']]

            #             g_df = pd.DataFrame.from_dict(record.features[0].qualifiers, orient='index').transpose()
            #             g_df = pd.DataFrame(g_df.iloc[0]).transpose()


                        #record taxonomy information (anything documented in source)
                        g_df.insert(1, 'accession', record.id)
                        g_df.insert(3, 'description', record.description)
            #             g_df['taxonomy'] = ';'.join(record.annotations['taxonomy'])
                        g_df['comment'] = record.annotations['comment']
                        g_df['genome_len'] = len(record.seq)
                        g_df['genome_seq'] = str(record.seq)

                        #continue parsing cds information
                        cds_df = []
                        for f in record.features:            
                            if f.type == 'CDS':

                                #get seq_nt information
                                q = pd.DataFrame.from_dict(f.qualifiers, orient='index').transpose()
                                q = pd.DataFrame(q.iloc[0]).transpose().reset_index(drop=True)
                                c_df = pd.concat([g_df[['virus_name','accession', 'genome_type', 'virus_tax_id', 'description']], 
                                                 q], axis=1, join="inner").reset_index(drop=True)
                                
                                seq_nt = str(f.extract(record.seq))
                                seq_aa = str(c_df.iloc[0].translation)                                    
                                if seq_aa[-1] =='*':
                                    seq_aa = seq_aa[0:-1]
                                
                                test_aa = str(f.extract(record.seq).translate())
                                if test_aa[-1] =='*':
                                    test_aa = test_aa[0:-1]
                                    
                                if seq_aa != test_aa:
                                    match = False
#                                     print(seq_aa)
#                                     print(test_aa)
                                else:
                                    match = True
                                    
                                    
                                c_df.insert(c_df.shape[1], 'seq_aa',  seq_aa)

                                c_df.insert(c_df.shape[1], 'seq_nt', seq_nt)
                                c_df.insert(c_df.shape[1], 'length_nt', len(seq_nt))
                                c_df.insert(4, 'type', f.type)
                                #c_df.insert(2, 'protein_id', f.qualifiers['protein_id'][0])
                                c_df.insert(5, 'location', str(f.location))
                                c_df.insert(6, 'strand', f.strand)

                                if 'db_xref' in c_df.columns:
                                    geneid = c_df.db_xref[0].split(':')[1]
                                    c_df.drop(['db_xref'], axis = 1)
                                else:
                                    geneid = ''
                                c_df.insert(7, 'gene_id', geneid)
                                c_df.insert(8, 'polypep?', 'polypro' in f.qualifiers['product'][0])
                                c_df.insert(9, 'length_aa', len(c_df['seq_aa'][0]))
                                c_df.insert(10, 'aa_match', match)
                                c_df.insert(11, 'start_codon', str(f.extract(record.seq))[0:3])


                                cds_df.append(c_df)

                            if f.type == 'mat_peptide':

                                #get seq_nt information
                                q = pd.DataFrame.from_dict(f.qualifiers, orient='index').transpose()
                                q = pd.DataFrame(q.iloc[0]).transpose()
                                c_df = pd.concat([g_df[['virus_name','accession', 'genome_type', 'virus_tax_id', 'description']], 
                                                 q], axis=1, join="inner").reset_index(drop = True)

                                seq_aa = str(f.extract(record.seq).translate())
                                match = True
                                if seq_aa[-1] =='*':
                                    seq_aa = seq_aa[0:-1]
                                c_df.insert(c_df.shape[1], 'seq_aa',  seq_aa)

                                c_df.insert(c_df.shape[1], 'seq_nt', str(f.extract(record.seq)))
                                c_df.insert(c_df.shape[1], 'length_nt', len(str(f.extract(record.seq))))
                                c_df.insert(4, 'type', f.type)
                                #c_df.insert(2, 'protein_id', f.qualifiers['protein_id'][0])
                                c_df.insert(5, 'location', str(f.location))
                                c_df.insert(6, 'strand', f.strand)
                                c_df.insert(8, 'polypep?', True)
                                c_df.insert(7, 'length_aa', len(c_df['seq_aa'][0]))
                                c_df.insert(10, 'aa_match', match)
                                c_df.insert(11, 'start_codon', '')

                                cds_df.append(c_df)


                        #compile CDS for each genome
                        if len(cds_df) > 1:
                            cds_df = pd.concat(cds_df, sort=False).reset_index(drop = True)
                            #record number of annotated proteins for each genome
                            g_df.insert(4, 'num_cds', sum(cds_df['type']=='CDS'))
                            g_df.insert(4, 'num_matpro', len(cds_df[cds_df['polypep?']!=True]))
                            cds_df_all.append(cds_df)

                        elif len(cds_df) == 1:
                            cds_df = cds_df[0]
                            #record number of annotated proteins for each genome
                            g_df.insert(4, 'num_cds', sum(cds_df['type']=='CDS'))
                            g_df.insert(4, 'num_matpro', len(cds_df[cds_df['polypep?']!=True]))
                            cds_df_all.append(cds_df)
                            


                        genome_df.append(g_df)
                        
            if len(genome_df)>0:

                print(f"Total number of sequences found in genbank file {gb_file} : {i+1}")
                print(f"Number of sequences derived from viruses with human hosts: {j}")
            
            

if len(genome_df)>1:
    genome_df = pd.concat(genome_df, sort=False).reset_index(drop = True)         
    cds_df_all = pd.concat(cds_df_all, sort=False).reset_index(drop = True)        

    display(genome_df.head())
    display(cds_df_all.head())

    print(f"TOTAL Number of sequences derived from viruses with human hosts: {len(genome_df)}")
    print(f"TOTAL Number of organisms derived from viruses with human hosts: {genome_df['virus_name'].nunique()}")

genome_df.to_csv('/lab/solexa_weissman/yhc/viral_smORF/data_analysis/virushostdb/genome_files/genome_df_raw.csv')



NC_005831
NC_001653
NC_006439
NC_006313
NC_006430
NC_001490
NC_016895
NC_063383
NC_003310
Total number of sequences found in genbank file viral.4.genomic.gbff : 526
Number of sequences derived from viruses with human hosts: 9
NC_001798
NC_017993
NC_017994
NC_017995
NC_017996
NC_017997
NC_029901
NC_037612
NC_037613
NC_029932
NC_029933
NC_036636
NC_030297
NC_030231
NC_027026
NC_026946
NC_027141
NC_027142
NC_027140
NC_027528
NC_027779
NC_029255
NC_027998
NC_028125


/usr/local/lib/python3.8/dist-packages/Bio/GenBank/Scanner.py:1794: BiopythonParserWarning: Structured comment not parsed for NC_028230. Is it malformed?
  warnings.warn(


NC_029122
NC_029123
NC_029124
NC_033781
NC_033725
NC_018102
NC_033830
NC_033831
NC_033847
NC_018136
NC_018137
NC_018138
NC_018401
NC_034253
NC_034254
NC_034255
NC_034256
NC_034261
NC_034262
NC_034263
NC_030447
NC_030448
NC_030922
NC_031284
NC_031285
NC_032682
NC_036604
NC_036605
NC_036606
NC_036877
NC_034385
NC_034386
NC_034391
NC_034387
NC_034497
NC_034498
NC_034505
NC_034477
NC_034478
NC_034489
NC_034479
NC_034480
NC_034490
NC_034443
NC_035474
NC_019023
NC_019027
NC_019028
NC_019026
NC_023874
NC_023888
NC_023891
NC_023984
NC_024070
NC_024118
NC_020106
NC_020439
NC_020440
NC_020441
NC_020446
NC_020447
NC_020448
NC_020442
NC_020443
NC_020445
NC_020444
NC_024472
NC_024494
NC_024495
NC_024496
NC_020498
NC_020805
NC_020806
NC_020810
NC_024691
NC_024694
NC_020890
NC_021242
NC_021243
NC_021244
NC_024888
NC_021483
NC_021568
NC_021541
NC_021542
NC_021543
NC_021544
NC_021545
NC_021546
NC_021547
NC_021549
NC_021550
NC_021551
NC_021548
NC_025114
NC_025347
NC_021928
NC_022089
NC_025255


/usr/local/lib/python3.8/dist-packages/Bio/Seq.py:2979: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


NC_025343
NC_022095
NC_025391


...

/usr/local/lib/python3.8/dist-packages/Bio/Seq.py:2979: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


NC_055523
NC_055181
NC_055182
NC_055183
NC_002645
Total number of sequences found in genbank file viral.2.genomic.gbff : 692
Number of sequences derived from viruses with human hosts: 28
Total number of sequences found in genbank file viral.3.genomic.gbff : 2845
Number of sequences derived from viruses with human hosts: 0


,virus_name,accession,virus_tax_id,description,num_matpro,num_cds,refseq_id_seg,segmented,genome_type,genome_composition,...,host_name,host_lineage,pmid,evidence,sample_type,source_organism,refseq_id_y,comment,genome_len,genome_seq
0,Human coronavirus NL63,NC_005831.2,277944,"Human Coronavirus NL63, complete genome",5.0,7.0,NC_005831,False,ssRNA(+),Viruses,...,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,NaN,UniProt,NaN,NaN,NC_005831,PROVISIONAL REFSEQ: This record has not yet be...,27553,CTTAAAGAATTTTTCTATCTATAGATAGAGAATTTTCTTATTTAGA...
1,Hepatitis delta virus,NC_001653.2,12475,"Hepatitis delta virus, complete genome",2.0,2.0,NC_001653,False,ssRNA(-),Viruses,...,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,"3184270, 3627276","Literature, UniProt",NaN,NaN,NC_001653,REVIEWED REFSEQ: This record has been curated ...,1682,ATGAGCCAAGTTCCGAACAAGGATTCGCGGGGAGGATAGATCAGCG...
2,Pichinde virus,NC_006439.1,2905917,"Pichinde virus large RNA segment, complete seq...",2.0,2.0,"NC_006439,NC_006447",True,ssRNA(+/-),Viruses,...,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,NaN,UniProt,NaN,NaN,NC_006439,PROVISIONAL REFSEQ: This record has not yet be...,6997,CGCACCGAGGATCCTAGGCATTTCTTGATCATGGAGGAATACGTTT...
3,Sabia virus,NC_006313.1,2907957,"Sabia virus, complete genome",2.0,2.0,"NC_006313,NC_006317",True,ssRNA(+/-),Viruses,...,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,NaN,UniProt,NaN,NaN,NC_006313,PROVISIONAL REFSEQ: This record has not yet be...,7133,GCGCACAGTGGATCCTAGGCATTGCTTTCACGCTTTGAGGTGAACC...
4,Parainfluenza virus 5,NC_006430.1,2905673,Parainfluenza virus 5 strain W3A,8.0,8.0,NC_006430,False,ssRNA(-),Viruses,...,Homo sapiens,Eukaryota; Opisthokonta; Metazoa; Eumetazoa; B...,"23807746, 26583313, 28138777, 31450796",Literature,NaN,NaN,NC_006430,PROVISIONAL REFSEQ: This record has not yet be...,15246,ACCAAGGGGAAAATGAAGTGGTGACTCAAATCATCGAAGACCCTCG...


,virus_name,accession,genome_type,virus_tax_id,type,location,strand,gene_id,polypep?,length_aa,...,citation,old_locus_tag,EC_number,function,exception,gene_synonym,trans_splicing,experiment,number,standard_name
0,Human coronavirus NL63,NC_005831.2,ssRNA(+),277944,CDS,"join{[286:12439](+), [12438:20475](+)}",1,2943501,True,6729,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Human coronavirus NL63,NC_005831.2,ssRNA(+),277944,CDS,[286:12469](+),1,2943501,True,4060,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Human coronavirus NL63,NC_005831.2,ssRNA(+),277944,CDS,[20471:24542](+),1,2943499,False,1356,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Human coronavirus NL63,NC_005831.2,ssRNA(+),277944,CDS,[24541:25219](+),1,2943500,False,225,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Human coronavirus NL63,NC_005831.2,ssRNA(+),277944,CDS,[25199:25433](+),1,2943502,False,77,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


TOTAL Number of sequences derived from viruses with human hosts: 813
TOTAL Number of organisms derived from viruses with human hosts: 502


In [9]:
orf_df = []


for j,acc in enumerate(genome_df.accession):
    

    ###################### Compiling ORF DataFrame
    #retrieve descriptors about the genome that the ORFs came from
    #retrieve all ORFs
    sub = genome_df.iloc[j]
    all_orfs = find_orfs_with_trans(Seq.Seq(sub.genome_seq)) 
    all_orfs.insert(0, 'virus_name', [sub['virus_name']]*len(all_orfs))
    all_orfs.insert(1, 'accession', [sub.accession]*len(all_orfs))
    all_orfs.insert(2, 'description', [sub.description]*len(all_orfs))
    all_orfs.insert(3, 'genome_type', [sub.genome_type]*len(all_orfs))
    all_orfs.insert(4, 'type', ['CDS']*len(all_orfs))
    all_orfs.insert(all_orfs.shape[1], 'virus_tax_id', [sub.virus_tax_id]*len(all_orfs))
    
    
    all_orfs.insert(4, 'orf_name', ['ORF_'+str(i+1) for i in range(len(all_orfs))])
    all_orfs.insert(9, 'aa_match', [True]*len(all_orfs))
    #all_orfs.insert(3, 'gene_len', [len(all_orfs.iloc[i]['seq_nt']) for i in range(len(all_orfs))])
    orf_df.append(all_orfs)

#filter out non-human host viruses
orf_df = pd.concat(orf_df).sort_values('length_aa').reset_index(drop=True)
display(orf_df.head(10))


# Export as csv
orf_df.to_csv(out_path + 'refseq_all_orf_df_raw.csv')



,virus_name,accession,description,genome_type,orf_name,type,location,start,end,aa_match,strand,seq_nt,seq_aa,length_aa,start_codon,virus_tax_id
0,Coxsackievirus B3,NC_038307.1,"Coxsackievirus B3 mRNA, complete genome",ssRNA(+),ORF_141,CDS,[5366:5393](+),5366,5393,True,1,CTGCCTTTGAGTTCGCTGTCGCAATGA,MPLSSLSQ,8,CTG,12072
1,Monkeypox virus,NC_003310.1,"Monkeypox virus Zaire-96-I-16, complete genome",dsDNA,ORF_2693,CDS,[149551:149578](+),149551,149578,True,1,ATGGTTGTTCACTTAATCCTAGTTTGA,MVVHLILV,8,ATG,10244
2,Human betaherpesvirus 6B,NC_000898.1,"Human herpesvirus 6B, complete genome",dsDNA,ORF_3263,CDS,[145151:145178](+),145151,145178,True,1,ATGCATGACCGCTACCTCACAGGGTAA,MHDRYLTG,8,ATG,32604
3,Human betaherpesvirus 6B,NC_000898.1,"Human herpesvirus 6B, complete genome",dsDNA,ORF_3237,CDS,[144474:144501](+),144474,144501,True,1,CTGCCTCCCACTCCACGGGGCCTTTGA,MPPTPRGL,8,CTG,32604
4,Human betaherpesvirus 6B,NC_000898.1,"Human herpesvirus 6B, complete genome",dsDNA,ORF_3156,CDS,[139958:139985](+),139958,139985,True,1,CTGATAAATTGTCATGTTTTTTTTTAA,MINCHVFF,8,CTG,32604
5,Dengue virus type 3,NC_001475.2,"Dengue virus 3, complete genome",ssRNA(+),ORF_592,CDS,[3246:3273](-),3246,3273,True,-1,ATGACAACTGTTGTTCCTTCACAGTAG,MTTVVPSQ,8,ATG,11069
6,Human betaherpesvirus 6B,NC_000898.1,"Human herpesvirus 6B, complete genome",dsDNA,ORF_3150,CDS,[139735:139762](+),139735,139762,True,1,CTGTATAATGTTAAACCGCAAGTGTAG,MYNVKPQV,8,CTG,32604
7,Dengue virus type 3,NC_001475.2,"Dengue virus 3, complete genome",ssRNA(+),ORF_612,CDS,[2474:2501](-),2474,2501,True,-1,CTGTCCAGGTATGGACCTCATTGGTGA,MSRYGPHW,8,CTG,11069
8,Human betaherpesvirus 6B,NC_000898.1,"Human herpesvirus 6B, complete genome",dsDNA,ORF_3269,CDS,[145367:145394](+),145367,145394,True,1,CTGGACTCTGAAACGGAACCCCCCTAA,MDSETEPP,8,CTG,32604
9,Dengue virus type 3,NC_001475.2,"Dengue virus 3, complete genome",ssRNA(+),ORF_644,CDS,[1406:1433](-),1406,1433,True,-1,ATGCCTGAGGTGTTATCTCAGCCGTGA,MPEVLSQP,8,ATG,11069


# Summary of annotations methods

In [10]:
#merge all unannotated orfs with annotated orfs

#####retrieve all orfs using the method I developed
orf_df= pd.read_csv(out_path + 'refseq_all_orf_df_raw.csv')

orf_df_raw = orf_df.reset_index(drop=True)
orf_df_raw['orf_index'] = orf_df_raw.index
orf_df_raw = orf_df_raw[['accession', 'seq_aa', 'orf_name', 'orf_index']]
#490706 ORFs identified using my method
print(f"Number ORFs identified using my method: {orf_df.seq_aa.nunique()}")

#####merge with orfs that are already annotated. match based on amino acid sequence
cds_df_all['genes_index'] = cds_df_all.index
#7387 total genes annotated; includes polyproteins and there peptide products
print(f"Number annotated genes in my dataset (includes polyproteins and there peptide products): {cds_df_all.seq_aa.nunique()}")

#7387 total CDS identified; 
print(f"Number annotated ORFs/CDS in my dataset: {cds_df_all[cds_df_all['type']=='CDS'].seq_aa.nunique()}")


#5719 total genes were identified with my method
id_annotated_aa = cds_df_all.merge(orf_df_raw, on = ['accession','seq_aa']).sort_values('orf_index')
id_annotated_aa  = id_annotated_aa.loc[id_annotated_aa.orf_index.drop_duplicates().index].reset_index(drop=True)
id_annotated_aa['ann'] = [True] * len(id_annotated_aa )
id_annotated_aa['matpro_id'] = id_annotated_aa['protein_id']
#print(id_annotated_aa.columns)
print('')
print(f"Number annotated ORFs/CDS in identified in my dataset: {id_annotated_aa.seq_aa.nunique()} ({id_annotated_aa.seq_aa.nunique()/cds_df_all[cds_df_all['type']=='CDS'].seq_aa.nunique()})")





Number ORFs identified using my method: 450342
Number annotated genes in my dataset (includes polyproteins and there peptide products): 6995
Number annotated ORFs/CDS in my dataset: 5921

Number annotated ORFs/CDS in identified in my dataset: 5421 (0.9155548049315994)


In [11]:
orf_df[orf_df.length_aa<105].seq_aa.nunique()

328639

In [12]:
# #####look at ones i missed using my method
#look at ones I missed in my size retriction
print('Take a look at ones that are not identified by my method and are <105aa')
small = cds_df_all[cds_df_all.length_aa < 105]
unid_annotated_aa=small[~small.seq_aa.isin(id_annotated_aa.seq_aa)].reset_index(drop=True)
unid_annotated_aa['ann'] = [True] * len(unid_annotated_aa )
unid_annotated_aa['matpro_id'] = unid_annotated_aa['protein_id']
matpep_annotated_aa = unid_annotated_aa[unid_annotated_aa.type=='mat_peptide']
splice = [True if ('splic' in str(unid_annotated_aa.loc[i]['note'])) | ('exon' in str(unid_annotated_aa.loc[i]['note'])) | ('Splic' in str(unid_annotated_aa.loc[i]['note'])) else False for i in range(len(unid_annotated_aa))] 
splice_annotated_aa = unid_annotated_aa[splice & (unid_annotated_aa.type!='mat_peptide')]
frame = [True if ('frameshift' in str(unid_annotated_aa.loc[i]['note'])) else False for i in range(len(unid_annotated_aa))] 
frame_annotated_aa = unid_annotated_aa[frame & (unid_annotated_aa.type!='mat_peptide')& ~np.array(splice)]
altstart = [False if (unid_annotated_aa.loc[i]['start_codon'] in ['ATG','CTG']) else True for i in range(len(unid_annotated_aa))] 
altstart_annotated_aa = unid_annotated_aa[altstart & ~np.array(frame) & (unid_annotated_aa.type!='mat_peptide')& ~np.array(splice)]

leftover = unid_annotated_aa[~unid_annotated_aa.genes_index.isin(matpep_annotated_aa.genes_index) 
                             & ~unid_annotated_aa.genes_index.isin(splice_annotated_aa.genes_index)
                            & ~unid_annotated_aa.genes_index.isin(frame_annotated_aa.genes_index)
                            & ~unid_annotated_aa.genes_index.isin(altstart_annotated_aa.genes_index)]
#display(leftover)

print(f"Number of annotated genes that meet my threshold (includes polyproteins and there peptide products): {small.seq_aa.nunique()}  ({small.seq_aa.nunique()/cds_df_all.seq_aa.nunique()}) ")
print(f"Number of annotated genes that meet my threshold (only ORFs and CDS): {small[small['type']=='CDS'].seq_aa.nunique()}  ({small[small['type']=='CDS'].seq_aa.nunique()/small.seq_aa.nunique()}) ")
print(f"Number of annotated genes NOT identified in my dataset (includes polyproteins and there peptide products): {unid_annotated_aa.seq_aa.nunique()}  ({unid_annotated_aa.seq_aa.nunique()/small.seq_aa.nunique()}) ")
print(f"Number of annotated genes Identified in my dataset (only ORFs and CDS): {id_annotated_aa[id_annotated_aa.length_aa < 105].seq_aa.nunique()}  ({id_annotated_aa[id_annotated_aa.length_aa < 105].seq_aa.nunique()/small[small['type']=='CDS'].seq_aa.nunique()}) ")
print(f"Ones that are mature peptides: {matpep_annotated_aa.seq_aa.nunique()} ({matpep_annotated_aa.seq_aa.nunique()/small[small['type']=='CDS'].seq_aa.nunique()})")
print(f"Ones that are formed by splicing: {splice_annotated_aa.seq_aa.nunique()} ({splice_annotated_aa.seq_aa.nunique()/small[small['type']=='CDS'].seq_aa.nunique()})")
print(f"Ones that are formed by frameshifting: {frame_annotated_aa.seq_aa.nunique()} ({frame_annotated_aa.seq_aa.nunique()/small[small['type']=='CDS'].seq_aa.nunique()})")
print(f"Ones that are formed by alternative start codon (not ATG or CTG): {altstart_annotated_aa.seq_aa.nunique()} ({altstart_annotated_aa.seq_aa.nunique()/small[small['type']=='CDS'].seq_aa.nunique()})")
print(f"Ones that are leftover: {leftover.seq_aa.nunique()} ({leftover.seq_aa.nunique()/small[small['type']=='CDS'].seq_aa.nunique()})")
print('')
print('')

cols = ['virus_name', 'accession', 'description', 'virus_tax_id', 'genome_type', 'type', 'matpro_id', 'ann', 'location', 'product',
       'note',  'aa_match', 'seq_nt', 'seq_aa', 'length_aa', 'start_codon']
print('')
print('')


print(f"Take a look at ones that I missed with my method.")
unid_annotated_aa=cds_df_all[~cds_df_all.seq_aa.isin(id_annotated_aa.seq_aa)].reset_index(drop=True)
unid_annotated_aa['ann'] = [True] * len(unid_annotated_aa )
unid_annotated_aa['matpro_id'] = unid_annotated_aa['protein_id']
matpep_annotated_aa = unid_annotated_aa[unid_annotated_aa.type=='mat_peptide']
splice = [True if ('splic' in str(unid_annotated_aa.loc[i]['note'])) | ('exon' in str(unid_annotated_aa.loc[i]['note'])) | ('Splic' in str(unid_annotated_aa.loc[i]['note'])) else False for i in range(len(unid_annotated_aa))] 
splice_annotated_aa = unid_annotated_aa[splice & (unid_annotated_aa.type!='mat_peptide')]
frame = [True if ('frameshift' in str(unid_annotated_aa.loc[i]['note'])) else False for i in range(len(unid_annotated_aa))] 
frame_annotated_aa = unid_annotated_aa[frame & (unid_annotated_aa.type!='mat_peptide')& ~np.array(splice)]
altstart = [False if (unid_annotated_aa.loc[i]['start_codon'] in ['ATG','CTG']) else True for i in range(len(unid_annotated_aa))] 
altstart_annotated_aa = unid_annotated_aa[altstart & ~np.array(frame) & (unid_annotated_aa.type!='mat_peptide')& ~np.array(splice)]

leftover = unid_annotated_aa[~unid_annotated_aa.genes_index.isin(matpep_annotated_aa.genes_index) 
                             & ~unid_annotated_aa.genes_index.isin(splice_annotated_aa.genes_index)
                            & ~unid_annotated_aa.genes_index.isin(frame_annotated_aa.genes_index)
                            & ~unid_annotated_aa.genes_index.isin(altstart_annotated_aa.genes_index)]

print(f"Number of annotated genes in in my dataset (includes polyprotein and their products): {cds_df_all.seq_aa.nunique()}")
print(f"Number of annotated genes in in my dataset (ORFs and CDS only): {cds_df_all[cds_df_all['type']=='CDS'].seq_aa.nunique()} ({cds_df_all[cds_df_all['type']=='CDS'].seq_aa.nunique()/cds_df_all.seq_aa.nunique()})")
print(f"Number of annotated genes in NOT identified in my dataset (includes polyproteins and there peptide products): {unid_annotated_aa.seq_aa.nunique()}  ({unid_annotated_aa.seq_aa.nunique()/cds_df_all.seq_aa.nunique()}) ")
print(f"Number of annotated genes in NOT identified in my dataset (ORFs/CDS only): {unid_annotated_aa[unid_annotated_aa['type']=='CDS'].seq_aa.nunique()}  ({unid_annotated_aa[unid_annotated_aa['type']=='CDS'].seq_aa.nunique()/cds_df_all[cds_df_all['type']=='CDS'].seq_aa.nunique()}) ")
print(f"Ones that are mature peptides: {matpep_annotated_aa.seq_aa.nunique()} ({matpep_annotated_aa.seq_aa.nunique()/cds_df_all[cds_df_all['type']=='CDS'].seq_aa.nunique()})")
print(f"Ones that are formed by splicing: {splice_annotated_aa.seq_aa.nunique()} ({splice_annotated_aa.seq_aa.nunique()/cds_df_all[cds_df_all['type']=='CDS'].seq_aa.nunique()})")
print(f"Ones that are formed by frameshifting: {frame_annotated_aa.seq_aa.nunique()} ({frame_annotated_aa.seq_aa.nunique()/cds_df_all[cds_df_all['type']=='CDS'].seq_aa.nunique()})")
print(f"Ones that are formed by alternative start codon (not ATG or CTG): {altstart_annotated_aa.seq_aa.nunique()} ({altstart_annotated_aa.seq_aa.nunique()/cds_df_all[cds_df_all['type']=='CDS'].seq_aa.nunique()})")
print(f"Ones that are leftover: {leftover.seq_aa.nunique()} ({leftover.seq_aa.nunique()/cds_df_all[cds_df_all['type']=='CDS'].seq_aa.nunique()})")
print('')
print('')

matpep_annotated_aa.sort_values(by = 'length_aa').to_csv(out_path +'ann_matpep.csv')
splice_annotated_aa.sort_values(by = 'length_aa').to_csv(out_path +'ann_splice.csv')
frame_annotated_aa.sort_values(by = 'length_aa').to_csv(out_path +'ann_frame.csv')
leftover.sort_values(by = 'length_aa').to_csv(out_path +'ann_leftover.csv')

cols = ['virus_name', 'accession', 'description', 'virus_tax_id', 'genome_type', 'type', 'matpro_id', 'ann', 'location', 'product',
       'note',  'aa_match', 'seq_nt', 'seq_aa', 'length_aa', 'start_codon']





Take a look at ones that are not identified by my method and are <105aa
Number of annotated genes that meet my threshold (includes polyproteins and there peptide products): 912  (0.13037884203002145) 
Number of annotated genes that meet my threshold (only ORFs and CDS): 679  (0.7445175438596491) 
Number of annotated genes NOT identified in my dataset (includes polyproteins and there peptide products): 265  (0.2905701754385965) 
Number of annotated genes Identified in my dataset (only ORFs and CDS): 647  (0.9528718703976435) 
Ones that are mature peptides: 233 (0.3431516936671576)
Ones that are formed by splicing: 13 (0.01914580265095729)
Ones that are formed by frameshifting: 1 (0.0014727540500736377)
Ones that are formed by alternative start codon (not ATG or CTG): 4 (0.005891016200294551)
Ones that are leftover: 14 (0.020618556701030927)




Take a look at ones that I missed with my method.
Number of annotated genes in in my dataset (includes polyprotein and their products): 6995
Num

In [13]:
#####incorporate everything i missed and unannotated orfs as one dataframe

#add required columns into annotated dataframes
orf_df['note'] =[''] * len(orf_df_raw)
orf_df['product'] = orf_df_raw.orf_name
orf_df['ann'] = [False] * len(orf_df_raw)
orf_df['matpro_id'] = ['Unannotated'] * len(orf_df_raw)
orf_df_ann = id_annotated_aa[cols].copy().reset_index(drop=True)
orf_df_un = orf_df[~orf_df_raw.orf_index.isin(id_annotated_aa.orf_index)][cols].copy().reset_index(drop=True)

# id_annotated_aa: annotated and identified by my method
# unid_annotated_aa: annotated but unidentified by my method
# orf_df_ann: annotated and identified by my method
# orf_df_un: unannotated and identified by my method
orf_df_annotated=pd.concat([id_annotated_aa.sort_values('length_aa').reset_index(drop=True), 
                            unid_annotated_aa.reset_index(drop=True), 
                            orf_df_un.reset_index(drop=True)], 
                           sort=False)[cols].reset_index(drop=True)
# #orf_df_annotated=orf_df_annotated.merge(tax_df, on ='accession').sort_values('length_aa').reset_index(drop=True)
display(orf_df_annotated.head(5))
print(len(orf_df_annotated))
# # #orf_df_ann = pd.DataFrame({'matpro_id':['Unannotated']*len(orf_df_raw)})
# # #orf_df_ann.loc[annotated_aa.orf_index,'matpro_id'] = annotated_aa.protein_id
# # orf_df_annotated=orf_df_raw.copy()
# orf_df_annotated['ann'] = orf_df_raw.orf_index.isin(annotated_aa.orf_index)
# orf_df_annotated['matpro_id'] = np.array(len(orf_df_raw), dtype = object)
# orf_df_annotated.loc[annotated_aa.orf_index, 'matpro_id'] = annotated_aa['protein_id']
# orf_df_annotated.loc[~orf_df_raw.orf_index.isin(annotated_aa.orf_index), 'matpro_id'] = 'Unannotated'

# orf_df_annotated.head()



,virus_name,accession,description,virus_tax_id,genome_type,type,matpro_id,ann,location,product,note,aa_match,seq_nt,seq_aa,length_aa,start_codon
0,NY_014 poxvirus,NC_035469.1,"NY_014 poxvirus strain 2013, complete genome",2025360,dsDNA,CDS,YP_009408449.1,True,[66354:66456](-),Virus entry/fusion complex component,NaN,True,ATGATAGCGGTTGTCATGTTCTTAATAGCATTCATGTTCTGTAGTT...,MIAVVMFLIAFMFCSWLSYSYLRPYINNKPLDN,33,ATG
1,Vaccinia virus,NC_006998.1,"Vaccinia virus, complete genome",10245,dsDNA,CDS,YP_910498.1,True,[59743:59851](-),hypothetical protein,NaN,True,ATGCTCGTCGTAATTATGTTTTTTATAGCGTTTGCCTTCTGTAGTT...,MLVVIMFFIAFAFCSWLSYSYLRPYISTKELNKSR,35,ATG
2,Monkeypox virus,NC_063383.1,"Monkeypox virus, complete genome",10244,dsDNA,CDS,YP_010377183.1,True,[57522:57630](-),"MV membrane, EFC component",NaN,True,ATGCTCGTCGTAATTATGTTTTTTATAGCGTTTGTCTTCTGTAGTT...,MLVVIMFFIAFVFCSWLSYSYLCPYISTKELNKSR,35,ATG
3,Vaccinia virus,NC_006998.1,"Vaccinia virus, complete genome",10245,dsDNA,CDS,YP_910501.1,True,[182355:182469](-),hypothetical protein,hypothetical protein; the poxviridae are envel...,True,ATGTTAAACTTCAGTTTATGTTTGTACCCCGTATTCATACTTAACA...,MLNFSLCLYPVFILNKLVLRTQSIILHTINNASIKNR,37,ATG
4,Vaccinia virus,NC_006998.1,"Vaccinia virus, complete genome",10245,dsDNA,CDS,YP_910500.1,True,[163144:163258](+),hypothetical protein,NaN,True,ATGTCTGGGATAGTAAAATCTATCATATTGAGCGGACCATCTGGTT...,MSGIVKSIILSGPSGLGKTAIAKRLWEYIWICGVPYH,37,ATG


490303


In [14]:
print(len(id_annotated_aa)) # annotated and identified by my method
print(len(unid_annotated_aa)) # annotated but unidentified by my method
print(len(orf_df_ann)) # annotated and identified by my method
print(len(orf_df_un)) # unannotated and identified by my method))

5684
1698
5684
482921


In [15]:
# check which ones don't have matching aa with translated
display(orf_df[~orf_df.aa_match])
test = orf_df_annotated[~orf_df_annotated.aa_match].reset_index(drop=True)
bad = test[[True if test.loc[i].seq_aa != Seq.Seq(test.loc[i].seq_nt).translate() else False for i in range(len(test))]]
#display(bad)

print(len(bad))


# for i in range(len(test)):
#     print(test.loc[i].seq_aa) 
#     print(Seq.Seq(test.loc[i].seq_nt).translate())
#     print('')
    
#looks like it's mostly ones with alt start codon usage (not ATG or CTG) so it's ok I will keep it anyway
bad.to_csv(out_path +'non_aa-match.csv')

#remove all with same amino acid sequence
print(len(orf_df_annotated))
orf_df_annotated= orf_df_annotated.loc[orf_df_annotated.seq_aa.drop_duplicates(keep='first').index].reset_index(drop = True)
print(len(orf_df_annotated))
orf_df_annotated.to_csv(out_path +'refseq_all_orf_df_assigned.csv')


,Unnamed: 0,virus_name,accession,description,genome_type,orf_name,type,location,start,end,...,strand,seq_nt,seq_aa,length_aa,start_codon,virus_tax_id,note,product,ann,matpro_id


41
490303


/usr/local/lib/python3.8/dist-packages/Bio/Seq.py:2979: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


451916


In [16]:
cds_df_all.sort_values(by = 'length_aa', ascending=True).reset_index(drop=True).to_csv(out_path + 'refseq_all_cds_annotated.csv')
cds_df_all[cds_df_all.length_aa < 105].sort_values(by = 'length_aa', ascending=True).reset_index(drop=True).to_csv(out_path + 'refseq_all_cds_annotated_SMALL.csv')

orf_df_annotated[orf_df_annotated.length_aa < 105].sort_values(by = 'length_aa', ascending=True).reset_index(drop=True).to_csv(out_path + 'refseq_all_small.csv')

